In [1]:
%%capture
%pip install -q sktime


In [2]:
# @title
# 1️⃣ نصب git و کلون کردن ریپازیتوری
!git clone https://github.com/google-research/timesfm.git
%cd timesfm

# 2️⃣ نصب PyTorch (GPU) یا JAX (Flax) و TimesFM
# برای PyTorch:
#!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install --upgrade torch==2.7.1 torchvision==0.22.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128
!pip install -e .[torch]

# اگر می‌خوای Flax/JAX استفاده کنی، به جای بالا:
# !pip install --upgrade "jax[cuda11_cudnn86]" -f https://storage.googleapis.com/jax-releases/jax_releases.html
# !pip install -e .[flax]

# اگر XReg نیاز داری:
# !pip install -e .[xreg]

# 3️⃣ تست نصب مدل با یک مثال ساده
import torch
import numpy as np
import timesfm

torch.set_float32_matmul_precision("high")

fatal: destination path 'timesfm' already exists and is not an empty directory.
/content/timesfm
Looking in indexes: https://download.pytorch.org/whl/cu128
Obtaining file:///content/timesfm
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for timesfm (pyproject.toml) ... done
  Created wheel for timesfm: filename=timesfm-3.0.2-0.editable-py3-none-any.whl size=9822 sha256=ee87c9d8675e21023b25f77246e039c6a374f8afa26a1f6d87be0fb6d93aa17f
  Stored in directory: /tmp/pip-ephem-wheel-cache-i6_nc2b7/wheels/61/fc/68/8410c5d0ea03d175d2d18c29227b259d8624df1128d5455b14
Successfully built timesfm
  Attempting uninstall: timesfm
    Found existing installation: timesfm 3.0.2
    Uninstalling timesfm-3.0.2:
      Successfully uninstalled timesfm-3.0.2


In [3]:
# @title
# بارگذاری مدل TimesFM 2.5 با PyTorch
model = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")

model.compile(
    timesfm.ForecastConfig(
        max_context=5000,
        max_horizon=1000,
        normalize_inputs=True,
        use_continuous_quantile_head=False,
        force_flip_invariance=True,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
import json
import os
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import MinMaxScaler
from sktime.performance_metrics.forecasting import MeanAbsoluteScaledError

In [5]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Configuration


Set paths and benchmark configuration.


In [6]:
CONFIG_PATH = '/content/drive/MyDrive/Models parameters JSON files/seasonalnaive.json'
OUTPUT_DIR = '/content/drive/MyDrive/results/json/TimesFM'
TRAIN_RATIO = 0.80
SHOW_PLOTS = True

with open(CONFIG_PATH, 'r', encoding='utf-8') as f:
    CONFIG = json.load(f)

DATASETS = CONFIG.get('datasets', [])
TOTAL_TARGETS = sum(len(dataset.get('targets', [])) for dataset in DATASETS)
total_models = TOTAL_TARGETS
os.makedirs(OUTPUT_DIR, exist_ok=True)


## Split data into training and test parts


In [7]:
# Cell 2: Split data into training and test parts
def split_data(dates, values):
    split_index = int(len(values) * TRAIN_RATIO)

    train_values = values[:split_index]
    test_values = values[split_index:]
    train_dates = dates.iloc[:split_index]
    test_dates = dates.iloc[split_index:]

    return train_dates, train_values, test_dates, test_values

## Metrics and JSON output

In [8]:
def scale_values(y_train, y_true, y_pred):
    y_train = np.asarray(y_train, dtype=np.float64).reshape(-1, 1)
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1, 1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1, 1)

    scaler = MinMaxScaler()
    scaler.fit(y_train)
    y_true_scaled = scaler.transform(y_true)
    y_pred_scaled = scaler.transform(y_pred)

    return y_true_scaled, y_pred_scaled


In [9]:
def calculate_metrics(train_values, test_values, predictions):
    test_array = np.asarray(test_values, dtype=np.float64)
    prediction_array = np.asarray(predictions, dtype=np.float64)

    mase = MeanAbsoluteScaledError()(
        test_array, prediction_array, y_train=train_values
    )

    # Unscaled errors use the original values.
    mae = float(mean_absolute_error(test_array, prediction_array))
    mse = float(mean_squared_error(test_array, prediction_array))
    rmse = float(np.sqrt(mse))

    # Scaled errors use a MinMaxScaler fitted only on the training values.
    test_scaled, predictions_scaled = scale_values(
        train_values, test_array, prediction_array
    )
    scaled_mae = float(mean_absolute_error(test_scaled, predictions_scaled))
    scaled_mse = float(mean_squared_error(test_scaled, predictions_scaled))
    scaled_rmse = float(np.sqrt(scaled_mse))

    return {
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'mase': float(mase),
        'mae_scaled': scaled_mae,
        'mse_scaled': scaled_mse,
        'rmse_scaled': scaled_rmse,
    }


## Time-series JSON output

In [10]:
def save_timeseries_full(
    log_path,
    model_name,
    full_dates,
    full_values,
    train_dates,
    train_values,
    test_dates,
    test_values,
    forecast_dates,
    forecast_values,
    metrics
):
    # Read previous results from the JSON file.
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8') as file:
            log_data = json.load(file)
    else:
        log_data = []

    # Create the result for this model.
    entry = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model': model_name,
        'full_series': {
            'dates': [str(date) for date in full_dates],
            'values': [float(value) for value in full_values]
        },
        'train_series': {
            'dates': [str(date) for date in train_dates],
            'values': [float(value) for value in train_values]
        },
        'test_series': {
            'dates': [str(date) for date in test_dates],
            'values': [float(value) for value in test_values]
        },
        'forecast': {
            'dates': [str(date) for date in forecast_dates],
            'prediction': [float(value) for value in forecast_values]
        },
        'metrics': {
            'mae': float(metrics['mae']),
            'mse': float(metrics['mse']),
            'rmse': float(metrics['rmse']),
            'mae_scaled': float(metrics['mae_scaled']),
            'mse_scaled': float(metrics['mse_scaled']),
            'rmse_scaled': float(metrics['rmse_scaled']),
            'mase': None if np.isnan(metrics['mase']) else float(metrics['mase'])
        }
    }

    # Add the new result and save it.
    log_data.append(entry)

    with open(log_path, 'w', encoding='utf-8') as file:
        json.dump(log_data, file, indent=4, ensure_ascii=False)


## Prepare one target column

In [11]:
# Cell 1: Prepare one target column
def prepare_target(data, dataset, target):
    dataset_name = dataset['dataset_name']
    time_column = dataset['time_column']
    target_column = target['column']

    target_data = data[[time_column, target_column]].copy()
    target_data[target_column] = pd.to_numeric(target_data[target_column])

    dates = target_data[time_column]
    values = target_data[target_column].to_numpy(dtype=float)

    return dataset_name, time_column, target_column, dates, values

## Create TimesFM forecast


In [12]:
def create_forecast(train_values, test_values):
    prediction_length = len(test_values)

    if prediction_length == 0:
        return np.array([], dtype=np.float32)

    point_forecast, _ = model.forecast(
        horizon=prediction_length,
        inputs=[np.asarray(train_values, dtype=np.float32)]
    )
    predictions = np.asarray(point_forecast[0], dtype=np.float32)

    if len(predictions) != prediction_length:
        raise RuntimeError(
            f'Prediction length ({len(predictions)}) does not match '
            f'test length ({prediction_length}).'
        )

    return predictions


## Forecast, evaluate, and save each target

In [13]:
# Cell 7: Run all steps for one target column
def run_target(data, dataset, target, model_number):
    dataset_name, time_column, target_column, dates, values = prepare_target(
        data, dataset, target
    )

    train_dates, train_values, test_dates, test_values = split_data(dates, values)
    predictions = create_forecast(train_values, test_values)
    metrics = calculate_metrics(train_values, test_values, predictions)

    print(f'\nModel {model_number}/{total_models}: {dataset_name} - {target_column}')
    print('MAE scaled:', round(metrics['mae_scaled'], 4))
    print('RMSE scaled:', round(metrics['rmse_scaled'], 4))
    print('MASE:', round(metrics['mase'], 4))

    show_forecast(
        train_dates, train_values, test_dates, test_values, predictions,
        dataset_name, time_column, target_column
    )

    save_result(
        dataset_name, target_column, target, dates, values,
        train_dates, train_values, test_dates, test_values,
        predictions, metrics
    )


## Plot target forecast

In [14]:
# Cell 5: Draw the forecast chart
def show_forecast(train_dates, train_values, test_dates, test_values, predictions,
                  dataset_name, time_column, target_column):
    if SHOW_PLOTS:
        plt.figure(figsize=(16, 6))
        plt.plot(train_dates, train_values, label='Training data')
        plt.plot(test_dates, test_values, label='Actual values')
        plt.plot(test_dates, predictions, '--', label='TimesFM forecast')
        plt.title(f'{dataset_name} - {target_column}')
        plt.xlabel(time_column)
        plt.ylabel(target_column)
        plt.legend()
        plt.grid()
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()



## Save target output

In [15]:
def save_result(dataset_name, target_column, target, dates, values,
                train_dates, train_values, test_dates, test_values,
                predictions, metrics):
    output_file = target['output_file']
    log_path = os.path.join(OUTPUT_DIR, output_file)
    model_name = f'TimesFM_{dataset_name}_{target_column}'

    save_timeseries_full(
        log_path, model_name, dates, values,
        train_dates, train_values, test_dates, test_values,
        test_dates, predictions, metrics
    )

    all_results.append({
        'dataset': dataset_name,
        'target': target_column,
        'model': model_name,
        **metrics
    })

    print('JSON saved:', log_path)



## Run all datasets and targets

In [17]:


# Cell 8: Run all datasets and target columns
all_results = []
completed_models = 0

for dataset in DATASETS:
    dataset_name = dataset['dataset_name']
    file_path = dataset['file_path']
    time_column = dataset['time_column']

    data = pd.read_csv(file_path)
    data[time_column] = pd.to_datetime(data[time_column])
    data = data.sort_values(time_column).reset_index(drop=True)

    print('\n' + '=' * 60)
    print('Dataset:', dataset_name)

    for target in dataset['targets']:
        completed_models += 1
        run_target(data, dataset, target, completed_models)

Output hidden; open in https://colab.research.google.com to view.

In [18]:
print('\n\n')
print('=' * 110)
print('ALL DATASETS / TARGETS COMPLETED')
print('=' * 110)
if len(all_results) > 0:
    summary_df = pd.DataFrame(all_results)
    print(summary_df.to_string(index=False))
else:
    print('No target was completed.')

print('\n')
print('=' * 90)
print('PROCESS FINISHED')
print('=' * 90)
print('Datasets requested:', len(DATASETS))
print('Targets requested:', total_models)
print('Targets completed:', len(all_results))
print('Output directory:', OUTPUT_DIR)
if len(all_results) != total_models:
    print('\nWARNING:')
    print('Not all requested targets completed successfully.')
    print('Requested:', total_models)
    print('Completed:', len(all_results))
else:
    print('ALL REQUESTED TARGETS COMPLETED.')





ALL DATASETS / TARGETS COMPLETED
dataset         target                        model           mae          mse          rmse      mase  mae_scaled  mse_scaled  rmse_scaled
   WB90             wt              TimesFM_WB90_wt      0.301308 1.184582e-01      0.344178  2.241924    0.110775    0.016011     0.126536
   WB90            toc             TimesFM_WB90_toc      0.036398 2.161049e-03      0.046487  0.425317    0.077443    0.009783     0.098909
   WB90    temperature     TimesFM_WB90_temperature      0.864187 1.072826e+00      1.035773  1.526691    0.091063    0.011912     0.109144
   WB90             ph              TimesFM_WB90_ph      0.066139 8.251805e-03      0.090839  0.923942    0.206686    0.080584     0.283873
   WB90       humidity        TimesFM_WB90_humidity      6.550519 7.063625e+01      8.404537  2.770870    0.139373    0.031977     0.178820
   WB90             do              TimesFM_WB90_do      0.065870 1.193801e-02      0.109261  0.944824    0.104555    0.0300